# Step 6 — Fit SUP-LR, SUP-MLP, SUP-METRIC (WSL)

This notebook is the anchor-fitting phase after the locked CodeT5 cache has been exported to ID-aligned NPZ. It reads `step6_lock.json` and `adapter_manifest.json`, verifies their hashes and the frozen raw inputs, then fits each anchor separately for Devign and ReVeal.

Protocol: fit on train; choose each model/seed threshold on validation by MCC; generate test scores for reporting only. Seeds are 42/43/44, with 42 as headline. SUP-LR and SUP-MLP use the frozen 768d CodeT5 embeddings. SUP-METRIC computes the 26 static features from frozen source code inside the runner. No CodeT5 encoding and no LLM calls happen here.

Expected outputs: `anchor_scores.csv`, `anchor_summary.json`, and a provenance `anchor_run_manifest.json` for each dataset.

In [1]:
from pathlib import Path
import csv
import hashlib
import json
import os
import subprocess
import sys
import numpy as np

ROOT = Path('/mnt/d/JUNIOR/LabThayTho/GRACE-summary')
STEP6 = ROOT / 'STEP6_supervised_anchors'
LOCK_PATH = STEP6 / 'step6_lock.json'
LOCK_SHA256 = '64fb2c99d9044548aaa07e949f1cd710580ff1b11571fdb833ddc18a56f80575'
ADAPTER_ROOT = STEP6 / 'cache' / 'anchor_npz'
ADAPTER_MANIFEST = ADAPTER_ROOT / 'adapter_manifest.json'
ADAPTER_MANIFEST_SHA256 = '898ede843e4e8ff33ea7a2f55cdb4da27a3e3bb0dfcfda2697b83ebddb596de9'
RUN_ROOT = STEP6 / 'runs' / 'anchor' / 'locked_step6_20261108'
assert ROOT.is_dir(), f'WSL repo path is unavailable: {ROOT}'

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(chunk_size), b''):
            h.update(block)
    return h.hexdigest()

assert sha256_file(LOCK_PATH) == LOCK_SHA256, 'Step 6 lock changed; stop and review.'
assert sha256_file(ADAPTER_MANIFEST) == ADAPTER_MANIFEST_SHA256, 'Adapter manifest changed; stop and review.'
LOCK = json.loads(LOCK_PATH.read_text(encoding='utf-8'))
ADAPTERS = json.loads(ADAPTER_MANIFEST.read_text(encoding='utf-8'))
assert ADAPTERS['lock_sha256'] == LOCK_SHA256 and ADAPTERS['encoder_rerun'] is False
assert ADAPTERS['lock_commit'] == 'e8f3cdc'
print('Lock and adapter manifest verified.')
print('Kernel Python:', sys.executable)
print('Run output root:', RUN_ROOT)

Lock and adapter manifest verified.
Kernel Python: /mnt/d/JUNIOR/LabThayTho/GRACE-summary/.venv-wsl/bin/python
Run output root: /mnt/d/JUNIOR/LabThayTho/GRACE-summary/STEP6_supervised_anchors/runs/anchor/locked_step6_20261108


In [2]:
# Verify the frozen inputs, unchanged anchor implementation, and every NPZ before fitting.
protocol = LOCK['protocol']
assert sha256_file(ROOT / protocol['protocol_file']) == protocol['protocol_sha256']
freeze = LOCK['frozen_data']['freeze_manifest']
assert sha256_file(ROOT / freeze['path']) == freeze['sha256']
runner = ROOT / protocol['anchor_implementation']
assert sha256_file(runner) == protocol['anchor_implementation_sha256'], 'Anchor runner changed; stop and review.'
adapter_by_key = {(x['dataset'], x['split']): x for x in ADAPTERS['exports']}
assert len(adapter_by_key) == 6
verified_inputs = {}
for dataset, ds_lock in LOCK['frozen_data']['datasets'].items():
    for split in ('train', 'validation', 'test'):
        raw = ds_lock['raw'][split]
        manifest = ds_lock['manifests'][split]
        assert sha256_file(ROOT / raw['path']) == raw['sha256'], f'{dataset}/{split}: raw hash mismatch'
        assert sha256_file(ROOT / manifest['path']) == manifest['sha256'], f'{dataset}/{split}: frozen manifest mismatch'
        item = adapter_by_key[dataset, split]
        adapter_path = ROOT / item['path']
        assert sha256_file(adapter_path) == item['npz_sha256'], f'{dataset}/{split}: NPZ hash mismatch'
        with np.load(adapter_path, allow_pickle=False) as data:
            ids = data['ids'].astype(str).tolist()
            X = data['X']
            assert X.shape == (item['rows'], protocol['codet5']['dimension'])
            assert X.dtype == np.float32 and np.isfinite(X).all()
        assert len(ids) == len(set(ids)) == item['rows']
        verified_inputs[dataset, split] = {'path': str(adapter_path), 'npz_sha256': item['npz_sha256'], 'rows': len(ids), 'shape': [len(ids), int(X.shape[1])]}
        print(f'PASS {dataset}/{split}: {len(ids)} aligned rows')
print('PASS: frozen raw, lock, runner, and all six embedding adapters are verified.')

PASS devign/train: 21854 aligned rows
PASS devign/validation: 2732 aligned rows
PASS devign/test: 2732 aligned rows
PASS reveal/train: 18187 aligned rows
PASS reveal/validation: 2273 aligned rows
PASS reveal/test: 2274 aligned rows
PASS: frozen raw, lock, runner, and all six embedding adapters are verified.


In [3]:
# Put the diagnostic kit on the module path for `python -m grace_diag.supervised_anchor`.
SRC = ROOT / '01_Author_Baseline_Unified' / 'diagnostic_kit' / 'src'
env = os.environ.copy()
env['PYTHONPATH'] = str(SRC) + (os.pathsep + env['PYTHONPATH'] if env.get('PYTHONPATH') else '')
probe = subprocess.run([sys.executable, '-c', 'import grace_diag.supervised_anchor'], cwd=ROOT, env=env, capture_output=True, text=True)
if probe.returncode != 0:
    raise RuntimeError('Cannot import anchor runner in this WSL kernel. Check the Step 6 Python environment and dependencies.\n' + probe.stderr)
print('Anchor runner import OK:', sys.executable)

Anchor runner import OK: /mnt/d/JUNIOR/LabThayTho/GRACE-summary/.venv-wsl/bin/python


In [4]:
def validate_outputs(dataset, output_dir):
    scores_path = output_dir / 'anchor_scores.csv'
    summary_path = output_dir / 'anchor_summary.json'
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    expected_arms = {'SUP-LR', 'SUP-MLP', 'SUP-METRIC'}
    expected_seeds = set(LOCK['protocol']['seeds'])
    assert len(summary) == 9
    assert {(r['arm'], r['seed']) for r in summary} == {(a, s) for a in expected_arms for s in expected_seeds}
    assert all(0.0 <= float(r['tau']) <= 1.0 for r in summary)
    counts = {'validation': 0, 'test': 0}
    unique = set()
    with scores_path.open(encoding='utf-8', newline='') as f:
        for row in csv.DictReader(f):
            assert row['dataset'] == dataset and row['split'] in counts
            assert row['arm'] in expected_arms and int(row['seed']) in expected_seeds
            assert np.isfinite(float(row['score']))
            key = (row['id'], row['split'], row['arm'], int(row['seed']))
            assert key not in unique, f'Duplicate output row: {key}'
            unique.add(key)
            counts[row['split']] += 1
    for split in counts:
        n = LOCK['embedding_cache_audit']['cache_used_for_final_prompt_retrieval']['datasets'][dataset]['splits'][split]['rows']
        assert counts[split] == n * len(expected_arms) * len(expected_seeds), f'{dataset}/{split}: unexpected score row count {counts[split]}'
    return scores_path, summary_path, counts

RUN_ROOT.mkdir(parents=True, exist_ok=True)
run_records = []
for dataset in ('devign', 'reveal'):
    output_dir = RUN_ROOT / dataset
    output_dir.mkdir(parents=True, exist_ok=True)
    scores_path = output_dir / 'anchor_scores.csv'
    summary_path = output_dir / 'anchor_summary.json'
    record_path = output_dir / 'anchor_run_manifest.json'
    if record_path.exists():
        old = json.loads(record_path.read_text(encoding='utf-8'))
        assert old['lock_sha256'] == LOCK_SHA256 and old['anchor_runner_sha256'] == protocol['anchor_implementation_sha256']
        assert sha256_file(scores_path) == old['anchor_scores_sha256']
        assert sha256_file(summary_path) == old['anchor_summary_sha256']
        print('Already complete under the same lock; reusing:', dataset)
    else:
        if scores_path.exists() or summary_path.exists():
            raise FileExistsError(f'Partial outputs exist for {dataset}; preserve them and use a new run directory after review: {output_dir}')
        ds = LOCK['frozen_data']['datasets'][dataset]
        args = [
            sys.executable, '-m', 'grace_diag.supervised_anchor', '--dataset', dataset,
            '--emb-train', str(ADAPTER_ROOT / dataset / 'train.npz'),
            '--emb-val', str(ADAPTER_ROOT / dataset / 'validation.npz'),
            '--emb-test', str(ADAPTER_ROOT / dataset / 'test.npz'),
            '--code-train', str(ROOT / ds['raw']['train']['path']),
            '--code-val', str(ROOT / ds['raw']['validation']['path']),
            '--code-test', str(ROOT / ds['raw']['test']['path']),
            '--out-dir', str(output_dir), '--seeds', ','.join(map(str, protocol['seeds']))
        ]
        print('Fitting', dataset, 'with locked seeds', protocol['seeds'])
        subprocess.run(args, cwd=ROOT, env=env, check=True)
    scores_path, summary_path, counts = validate_outputs(dataset, output_dir)
    execution = {
        'schema': 'step6_anchor_run_manifest_v1', 'dataset': dataset,
        'lock_path': str(LOCK_PATH.relative_to(ROOT)), 'lock_sha256': LOCK_SHA256,
        'lock_commit': 'e8f3cdc', 'anchor_runner': str(runner.relative_to(ROOT)),
        'anchor_runner_sha256': sha256_file(runner), 'seeds': protocol['seeds'],
        'headline_seed': protocol['headline_seed'],
        'threshold_rule': 'MCC selected on validation only; test labels not used for fit or threshold selection',
        'inputs': {split: verified_inputs[dataset, split] for split in ('train', 'validation', 'test')},
        'score_rows': counts, 'anchor_scores_sha256': sha256_file(scores_path),
        'anchor_summary_sha256': sha256_file(summary_path)
    }
    record_path.write_text(json.dumps(execution, indent=2) + '\n', encoding='utf-8')
    run_records.append(execution)
    print(f'PASS {dataset}: {counts["validation"]} validation scores; {counts["test"]} test scores; summary has 9 arm/seed rows')
print('Both dataset anchor runs completed/verified:', RUN_ROOT)

Fitting devign with locked seeds [42, 43, 44]
wrote /mnt/d/JUNIOR/LabThayTho/GRACE-summary/STEP6_supervised_anchors/runs/anchor/locked_step6_20261108/devign
PASS devign: 24588 validation scores; 24588 test scores; summary has 9 arm/seed rows
Fitting reveal with locked seeds [42, 43, 44]
wrote /mnt/d/JUNIOR/LabThayTho/GRACE-summary/STEP6_supervised_anchors/runs/anchor/locked_step6_20261108/reveal
PASS reveal: 20457 validation scores; 20466 test scores; summary has 9 arm/seed rows
Both dataset anchor runs completed/verified: /mnt/d/JUNIOR/LabThayTho/GRACE-summary/STEP6_supervised_anchors/runs/anchor/locked_step6_20261108


## After this notebook

Review both `anchor_summary.json` files and the `anchor_run_manifest.json` checksums. Do not choose a new threshold from test results. Once the B2 score files are available and their IDs/split/protocol match, proceed to the paired P_pre/P_post and complementarity/routing analyses.